In [1]:
# cell 1
# Mount Google Drive and set project workspace.

import os
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

def find_baseline_dir():
    candidates = [
        "/content/drive/MyDrive/final_project/baseline",
        "/content/drive/MyDrive/final_project/baseline/",
    ]

    for p in candidates:
        if os.path.isdir(p):
            return os.path.abspath(p)

    shared_root = "/content/drive/Shareddrives"
    if os.path.isdir(shared_root):
        for root, dirs, _ in os.walk(shared_root):
            if root.endswith("/final_project") and "baseline" in dirs:
                return os.path.abspath(os.path.join(root, "baseline"))

    raise FileNotFoundError("Could not find final_project/baseline in Drive.")

BASE_DIR = find_baseline_dir()
os.chdir(BASE_DIR)

print("BASE_DIR =", BASE_DIR)
print("CWD =", os.getcwd())

Mounted at /content/drive
BASE_DIR = /content/drive/MyDrive/final_project/baseline
CWD = /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline


In [2]:
#cell 2
# Install only evaluation dependencies.

import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-U",
    "pip",
    "setuptools",
    "wheel",
])

pkgs = [
    "sentence-transformers",
    "tqdm==4.66.2",
    "pandas",
]

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
] + pkgs)

print("Installed OK")

Installed OK


In [3]:
# cell 3
# Use CUDA for Colab L4 GPU.

import torch

assert torch.cuda.is_available(), "GPU is required. Please enable GPU in Colab."

DEVICE = "cuda"

print("DEVICE =", DEVICE)
print("GPU =", torch.cuda.get_device_name(0))
print("GPU memory GB =", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

DEVICE = cuda
GPU = NVIDIA L4
GPU memory GB = 22.03


In [4]:
# cell 4
# Define evidence paths.

import os

HOTpot_EVIDENCE_PATH = os.path.join(
    BASE_DIR,
    "DATA",
    "KG",
    "evidence",
    "hotpot_evidence_1000",
    "qwen_agent",
    "evidence.json",
)

TWO_WIKI_EVIDENCE_PATH = os.path.join(
    BASE_DIR,
    "DATA",
    "KG",
    "evidence",
    "2wikimultihopqa_evidence_1000",
    "qwen_agent",
    "evidence.json",
)

DATASET_CONFIGS = {
    "hotpotqa": {
        "path": HOTpot_EVIDENCE_PATH,
        "expected_types": ["bridge", "comparison"],
    },
    "2wikimultihopqa": {
        "path": TWO_WIKI_EVIDENCE_PATH,
        "expected_types": [
            "bridge_comparison",
            "inference",
            "comparison",
            "compositional",
        ],
    },
}

for dataset_name, cfg in DATASET_CONFIGS.items():
    print(dataset_name, "=>", cfg["path"])
    assert os.path.isfile(cfg["path"]), f"Missing evidence file: {cfg['path']}"

print("All evidence files exist.")

hotpotqa => /content/drive/MyDrive/final_project/baseline/DATA/KG/evidence/hotpot_evidence_1000/qwen_agent/evidence.json
2wikimultihopqa => /content/drive/MyDrive/final_project/baseline/DATA/KG/evidence/2wikimultihopqa_evidence_1000/qwen_agent/evidence.json
All evidence files exist.


In [5]:
# cell 5
# These labels separate the reports per model.
# The GitHub EM logic evaluates evidence.json, not model response files.

MODEL_NAMES = [
    "qwen3_5_9b",
    "gpt_oss_120b",
]

print("Models:")
for model_name in MODEL_NAMES:
    print("-", model_name)

Models:
- qwen3_5_9b
- gpt_oss_120b


In [6]:
# cell 6
# Load the same embedding model used in the GitHub code.

import json
import pandas as pd

from tqdm import tqdm
from collections import Counter, defaultdict
from sentence_transformers import util
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"
THRESHOLD = 0.9

emb = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Threshold:", THRESHOLD)
print("Device:", DEVICE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Threshold: 0.9
Device: cuda


In [7]:
# cell 7
# Load JSON files.

def load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

print("JSON loader ready.")

JSON loader ready.


In [8]:
# cell 8
# GitHub-style support-level EM.
# A support is correct if max dot_score with found evidence is greater than 0.9.

def evaluate_em_github_style(data, dataset_name, threshold=0.9):
    total_supports = 0
    correct_supports = 0

    total_by_type = defaultdict(int)
    correct_by_type = defaultdict(int)

    for record in tqdm(data, total=len(data), desc=f"Evaluating {dataset_name}"):
        q_type = record["type"]
        found_evidence = record["evidence"]
        supports = record["supports"]

        evidence_emb = emb.encode(
            found_evidence,
            device=DEVICE,
        )

        for s in supports:
            total_supports += 1
            total_by_type[q_type] += 1

            support_emb = emb.encode(
                s[1],
                device=DEVICE,
            )

            sim_score = util.dot_score(
                support_emb,
                evidence_emb,
            ).cpu().numpy().flatten()

            if max(sim_score) > threshold:
                correct_supports += 1
                correct_by_type[q_type] += 1

    overall_em = correct_supports / total_supports if total_supports else 0.0

    results = {
        "overall": {
            "total_supports": total_supports,
            "correct_supports": correct_supports,
            "em": overall_em,
        },
        "by_type": {},
    }

    for q_type in sorted(total_by_type.keys()):
        total = total_by_type[q_type]
        correct = correct_by_type[q_type]
        em = correct / total if total else 0.0

        results["by_type"][q_type] = {
            "total_supports": total,
            "correct_supports": correct,
            "em": em,
        }

    return results

print("GitHub-style EM evaluator ready.")

GitHub-style EM evaluator ready.


In [9]:
# cell 9
# Inspect dataset sizes and question types.

loaded_datasets = {}

for dataset_name, cfg in DATASET_CONFIGS.items():
    data = load_json(cfg["path"])
    loaded_datasets[dataset_name] = data

    print("=" * 80)
    print("DATASET:", dataset_name)
    print("Path:", cfg["path"])
    print("Records:", len(data))
    print("Type counts:", Counter(record["type"] for record in data))
    print("First keys:", list(data[0].keys()))
    print("First question:", data[0]["question"])

DATASET: hotpotqa
Path: /content/drive/MyDrive/final_project/baseline/DATA/KG/evidence/hotpot_evidence_1000/qwen_agent/evidence.json
Records: 1000
Type counts: Counter({'bridge': 700, 'comparison': 300})
First keys: ['type', 'question', 'evidence', 'answer', 'supports']
First question: Where operation Operation Dragoon and Battle of Cold Harbor fought during to different wars?
DATASET: 2wikimultihopqa
Path: /content/drive/MyDrive/final_project/baseline/DATA/KG/evidence/2wikimultihopqa_evidence_1000/qwen_agent/evidence.json
Records: 1000
Type counts: Counter({'bridge_comparison': 250, 'inference': 250, 'comparison': 250, 'compositional': 250})
First keys: ['type', 'question', 'evidence', 'answer', 'supports']
First question: Do both films: And Then There Were None (1945 Film) and Langue Sacrée, Langue Parlée have the directors from the same country?


In [10]:
# cell 10
# Run separate reports for each model and dataset.
# Since this GitHub metric uses evidence.json, results can be identical if evidence paths are identical.

all_rows = []

for model_name in MODEL_NAMES:
    print("\n" + "#" * 100)
    print("MODEL:", model_name)
    print("#" * 100)

    for dataset_name, data in loaded_datasets.items():
        print("\n" + "=" * 100)
        print("DATASET:", dataset_name)
        print("=" * 100)

        results = evaluate_em_github_style(
            data=data,
            dataset_name=dataset_name,
            threshold=THRESHOLD,
        )

        overall = results["overall"]

        print("\nOverall EM:")
        print(
            f"Total supports: {overall['total_supports']} | "
            f"Correct supports: {overall['correct_supports']} | "
            f"EM: {overall['em']:.4f}"
        )

        all_rows.append({
            "model": model_name,
            "dataset": dataset_name,
            "split": "overall",
            "total_supports": overall["total_supports"],
            "correct_supports": overall["correct_supports"],
            "em": overall["em"],
        })

        print("\nEM by question type:")
        for q_type, type_result in results["by_type"].items():
            print(
                f"{q_type} | "
                f"Total supports: {type_result['total_supports']} | "
                f"Correct supports: {type_result['correct_supports']} | "
                f"EM: {type_result['em']:.4f}"
            )

            all_rows.append({
                "model": model_name,
                "dataset": dataset_name,
                "split": q_type,
                "total_supports": type_result["total_supports"],
                "correct_supports": type_result["correct_supports"],
                "em": type_result["em"],
            })


####################################################################################################
MODEL: qwen3_5_9b
####################################################################################################

DATASET: hotpotqa


Evaluating hotpotqa: 100%|██████████| 1000/1000 [00:35<00:00, 27.98it/s]



Overall EM:
Total supports: 2400 | Correct supports: 775 | EM: 0.3229

EM by question type:
bridge | Total supports: 1724 | Correct supports: 483 | EM: 0.2802
comparison | Total supports: 676 | Correct supports: 292 | EM: 0.4320

DATASET: 2wikimultihopqa


Evaluating 2wikimultihopqa: 100%|██████████| 1000/1000 [00:40<00:00, 24.97it/s]



Overall EM:
Total supports: 2501 | Correct supports: 921 | EM: 0.3683

EM by question type:
bridge_comparison | Total supports: 1000 | Correct supports: 318 | EM: 0.3180
comparison | Total supports: 501 | Correct supports: 320 | EM: 0.6387
compositional | Total supports: 500 | Correct supports: 151 | EM: 0.3020
inference | Total supports: 500 | Correct supports: 132 | EM: 0.2640

####################################################################################################
MODEL: gpt_oss_120b
####################################################################################################

DATASET: hotpotqa


Evaluating hotpotqa: 100%|██████████| 1000/1000 [00:34<00:00, 28.73it/s]



Overall EM:
Total supports: 2400 | Correct supports: 775 | EM: 0.3229

EM by question type:
bridge | Total supports: 1724 | Correct supports: 483 | EM: 0.2802
comparison | Total supports: 676 | Correct supports: 292 | EM: 0.4320

DATASET: 2wikimultihopqa


Evaluating 2wikimultihopqa: 100%|██████████| 1000/1000 [00:40<00:00, 25.00it/s]


Overall EM:
Total supports: 2501 | Correct supports: 921 | EM: 0.3683

EM by question type:
bridge_comparison | Total supports: 1000 | Correct supports: 318 | EM: 0.3180
comparison | Total supports: 501 | Correct supports: 320 | EM: 0.6387
compositional | Total supports: 500 | Correct supports: 151 | EM: 0.3020
inference | Total supports: 500 | Correct supports: 132 | EM: 0.2640


In [11]:
# cell 11
# Display all EM results in one table.

em_df = pd.DataFrame(all_rows)

em_df["em"] = em_df["em"].round(4)

em_df = em_df.sort_values(
    by=["model", "dataset", "split"],
    key=lambda col: col.map(lambda x: "000_overall" if x == "overall" else str(x))
    if col.name == "split"
    else col,
).reset_index(drop=True)

display(em_df)

,model,dataset,split,total_supports,correct_supports,em
0,gpt_oss_120b,2wikimultihopqa,overall,2501,921,0.3683
1,gpt_oss_120b,2wikimultihopqa,bridge_comparison,1000,318,0.3180
2,gpt_oss_120b,2wikimultihopqa,comparison,501,320,0.6387
3,gpt_oss_120b,2wikimultihopqa,compositional,500,151,0.3020
4,gpt_oss_120b,2wikimultihopqa,inference,500,132,0.2640
5,gpt_oss_120b,hotpotqa,overall,2400,775,0.3229
6,gpt_oss_120b,hotpotqa,bridge,1724,483,0.2802
7,gpt_oss_120b,hotpotqa,comparison,676,292,0.4320
8,qwen3_5_9b,2wikimultihopqa,overall,2501,921,0.3683
9,qwen3_5_9b,2wikimultihopqa,bridge_comparison,1000,318,0.3180


In [12]:
# cell 12
# Print a clean grouped report.

for model_name in MODEL_NAMES:
    print("\n" + "#" * 100)
    print("MODEL:", model_name)
    print("#" * 100)

    model_df = em_df[em_df["model"] == model_name]

    for dataset_name in DATASET_CONFIGS.keys():
        print("\nDATASET:", dataset_name)

        dataset_df = model_df[model_df["dataset"] == dataset_name]

        for _, row in dataset_df.iterrows():
            print(
                f"{row['split']}: "
                f"Total supports: {int(row['total_supports'])} | "
                f"Correct supports: {int(row['correct_supports'])} | "
                f"EM: {row['em']:.4f}"
            )


####################################################################################################
MODEL: qwen3_5_9b
####################################################################################################

DATASET: hotpotqa
overall: Total supports: 2400 | Correct supports: 775 | EM: 0.3229
bridge: Total supports: 1724 | Correct supports: 483 | EM: 0.2802
comparison: Total supports: 676 | Correct supports: 292 | EM: 0.4320

DATASET: 2wikimultihopqa
overall: Total supports: 2501 | Correct supports: 921 | EM: 0.3683
bridge_comparison: Total supports: 1000 | Correct supports: 318 | EM: 0.3180
comparison: Total supports: 501 | Correct supports: 320 | EM: 0.6387
compositional: Total supports: 500 | Correct supports: 151 | EM: 0.3020
inference: Total supports: 500 | Correct supports: 132 | EM: 0.2640

####################################################################################################
MODEL: gpt_oss_120b
#########################################################